# Milestone 4 lab, notebook 5 of 8: Warn, block or set aside

## Where we are

| Notebook | What you do |
|---|---|
| 1. A rule lives in dbt | Move a check out of a notebook and into a file that dbt runs |
| 2. How serious is it | Tell dbt which failures stop the night. Keep the rows a test returns |
| 3. Test the SQL, not the data | Check your own query on four rows typed by hand |
| 4. A gate, and where it stands | Run one rule in three places, and see what each place can see |
| **5. Warn, block or set aside** (you are here) | Decide what a failure does. Then block a real run in Airflow |
| 6. Quarantine | Set the bad rows aside with a reason, and let the good rows go on |
| 7. Test the tests | Plant a fault, and prove its gate catches it. One check in four tools |
| 8. The production version | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. The image has changed since milestone 1: it now holds dbt, so the first start builds for a minute or two.

**About ten minutes.** Two runs in Airflow take about a minute of it.

In [1]:
import pandas as pd
from lab_helpers import all_tests, change, money, prepare_week, sql

import airflow_tools  # lab_helpers put this milestone's folder on the path, so this is found

prepare_week()
airflow_tools.wait_until_ready()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August, through three gates.
Sunday night ran 47 tests. 1 of 6 stores can be paid.


## Step 1: A check that fails must do something

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

The head of retail operations signs the bonus on Monday. In milestone 1 she signed six wrong numbers, because a failed check could only do one thing: nothing.

A gate has three choices.

| Choice | What happens | The price |
|---|---|---|
| **Warn** | The run goes on. The warning is recorded | A wrong value may reach a reader |
| **Set aside** | The bad rows are taken out, with the reason. The good rows go on | The output is missing something, and must say so |
| **Block** | The gate fails. Nothing after it runs. The outputs keep their last good numbers | Nobody gets tonight's numbers, for any store |

The first and the last are the classic pair: **warn or block**. Setting rows aside is the middle way, and it is notebook 6.

## Step 2: You decide

Six checks. Before you run the next cell, say for each one: warn, set aside, or block?

1. Store `S04` charged yesterday's price for butter. The store is $6.00 short.
2. A refund has a positive amount.
3. The stores table has five rows where six are expected.
4. One store's day has more than twice its normal number of lines.
5. One person is in the members table twice.
6. A product has two prices on the same day.

Here is what the project decided. The `action` is written in each test's file.

In [2]:
tests = all_tests()
asked = ["price_of_the_day", "refund_gives_money_back", "six_stores_once", "normal_volume",
         "one_person_once", "one_price_at_a_time"]
tests[tests["test"].isin(asked)].set_index("test").loc[asked, ["action", "rule"]]

,action,rule
test,,
price_of_the_day,warn,The price charged is the price valid that day
refund_gives_money_back,quarantine,A refund gives money back
six_stores_once,block,"There are six stores, each once"
normal_volume,warn,A store's day is near its normal day
one_person_once,quarantine,One person is one member
one_price_at_a_time,block,A product has one price on any day


The reasoning behind each:

- **The butter: warn.** The published number is right. The till really took that money. Stopping the night would hurt five other stores to report six dollars.
- **The refund: set aside.** One row is bad. The other 112 lines of that store's day are fine, and so are five other stores.
- **Five stores: block.** Every sales line is joined to this table. Wrong here is wrong everywhere, and no row can be "set aside" to fix it.
- **A day twice its size: warn.** It is a reason to look. It is not a proof.
- **The same person twice: set aside** the newer one, so she gets one email.
- **Two prices: block.** Every sale of that product would be counted twice, in every store.

The rule behind the rules: **block when publishing would show wrong numbers and no row can be taken out to fix it. Warn when the numbers are still right.**

In [3]:
tests.groupby(["gate", "action"]).size().unstack(fill_value=0)

action,block,quarantine,refuse_file,warn
gate,,,,
clean,9,17,0,9
landing,0,0,1,1
publish,2,0,0,3


Eleven of these test files can block, and so can four generic tests in `sources.yml`. On a normal night none of them fails. You have to break something to see one.

## Step 3: Block a real run

Break the stores table. In the newest stores file, delete store `S06`. That is what "someone removed a row by mistake" looks like.

First remember what is published now.

In [4]:
published_before = sql("SELECT count(*) AS rows, sum(net_sales_cents) AS cents FROM output.daily_store_sales")
print("Published now:", published_before.loc[0, "rows"], "rows,", money(published_before.loc[0, "cents"]))

change("""
    DELETE FROM raw.stores
    WHERE store_id = 'S06' AND _file_name = (SELECT max(_file_name) FROM raw.stores)
""")
sql("SELECT _file_name, count(*) AS stores FROM raw.stores GROUP BY _file_name ORDER BY _file_name DESC LIMIT 2")

Published now: 208 rows, $159,368.94


,_file_name,stores
0,stores_2026-08-30.csv,5
1,stores_2026-08-29.csv,6


Five stores in tonight's file. Now let Airflow run Sunday's night. It takes about half a minute.

In [5]:
states = airflow_tools.run_night("2026-08-30")
pd.DataFrame(states)

,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,failed
5,build,upstream_failed
6,gate_publish,upstream_failed
7,publish,upstream_failed


Read the `state` column from the top.

- Four tasks are `success`. The files landed, and the landing gate had nothing to say.
- `gate_clean` is **`failed`**. A test whose action is `block` found a row.
- `build`, `gate_publish` and `publish` are **`upstream_failed`**. Airflow did not start them.

**Upstream** means earlier in the DAG. An **upstream failure** is a task that failed earlier. Airflow does not start the tasks after it, and marks them `upstream_failed`. You read about that state in milestone 1. Now you have made it happen.

What did the gate see?

In [6]:
sql("""
    SELECT gate, test_name, action, status, bad_rows, title
    FROM ops.gate_results
    WHERE run_day = DATE '2026-08-30' AND status <> 'pass' AND action = 'block'
""")

,gate,test_name,action,status,bad_rows,title
0,clean,six_stores_once,block,fail,1,"There are six stores, each once"
1,clean,every_store_is_known,block,fail,3,Every file is from a store we know


Two tests blocked. `six_stores_once` counted five stores, where six are expected. And `every_store_is_known` found files from a store that is no longer in the table: `S06` still sends sales lines, till totals and stock counts.

**Look at it yourself:** open http://localhost:8080 , click **Dags**, then `lotus_nightly_m4`. The newest run has one red box and three orange ones.

## Step 4: The last good numbers

`publish` did not run. So what do the readers see this morning?

In [7]:
published_after = sql("SELECT count(*) AS rows, sum(net_sales_cents) AS cents FROM output.daily_store_sales")
print("Published before:", published_before.loc[0, "rows"], "rows,", money(published_before.loc[0, "cents"]))
print("Published after: ", published_after.loc[0, "rows"], "rows,", money(published_after.loc[0, "cents"]))
print("The same:", published_before.equals(published_after))

Published before: 208 rows, $159,368.94
Published after:  208 rows, $159,368.94
The same: True


Exactly what they saw yesterday. The table people read was not touched. It is a day old, and it is right. That is what "the outputs keep their last good numbers" means.

It works because of how this pipeline publishes. It does not build the outputs in the place people read. It builds them in a side room first.

In [8]:
sql("""
    SELECT table_schema, table_name FROM information_schema.tables
    WHERE table_schema IN ('staging', 'output') ORDER BY table_schema DESC, table_name
""")

,table_schema,table_name
0,staging,daily_store_sales
1,staging,replenishment_list
2,output,daily_store_sales
3,output,replenishment_list
4,output,store_day_status


The schema `staging` is the side room. Three steps:

1. **Write.** The `build` task writes tonight's outputs into `staging`. Nobody reads `staging`.
2. **Audit.** The `gate_publish` task runs its tests on the staging tables.
3. **Publish.** Only if the audit passes, the `publish` task copies them over the tables in `output`, in one step.

This pattern is called **write, audit, publish**. Its opposite is building straight into the table people read, and checking afterwards. Then a failed check finds the wrong numbers already on the manager's screen.

## Step 5: Put it back

A block is not fixed by running again. The same test would fail again. Somebody must fix the cause.

Here the cause is one damaged file. Take that file's rows out of the raw tables, so the load step loads it again from the stores.

In [9]:
newest = sql("SELECT max(_file_name) AS file FROM raw.stores").loc[0, "file"]
change("DELETE FROM raw.stores WHERE _file_name = ?", (newest,))
change("DELETE FROM raw.loaded_files WHERE file_name = ?", (newest,))
change("DELETE FROM raw.file_columns WHERE file_name = ?", (newest,))
print("Forgotten:", newest)

states = airflow_tools.run_night("2026-08-30")
pd.DataFrame(states)

Forgotten: stores_2026-08-30.csv


,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [10]:
sql("SELECT _file_name, count(*) AS stores FROM raw.stores GROUP BY _file_name ORDER BY _file_name DESC LIMIT 1")

,_file_name,stores
0,stores_2026-08-30.csv,6


Eight green tasks, and six stores again. The warehouse is as the nightly run left it.

## What you now know

- A failed check must do something: **warn or block**, or set rows aside.
- Block when publishing would show wrong numbers and no row can be taken out. Warn when the numbers are still right.
- An **upstream failure** is a task that failed earlier in the DAG. Airflow marks the tasks after it `upstream_failed` and does not start them.
- **Write, audit, publish**: build the output in a side table, check it there, and only then show it to its readers.
- After a block, the readers keep the last good numbers. A day old, and right.

**Say it in an interview:** "We built outputs in a staging schema, audited them there, and published only on a pass. When a gate blocked, Airflow marked the tasks after it upstream_failed, and the published tables kept the last good numbers. I chose block only where a wrong reference table would make every number wrong."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), sections "Write, audit, publish" and "Warn or block". For the task states: [Airflow: Branching, Trigger Rules & Task Groups](https://www.d8loop.com/modules/knowledge-corner/airflow/branching-trigger-rules-and-task-groups).

Next: [notebook 6](06_quarantine.ipynb). Blocking six stores for one bad row is too much. Set the row aside.